# Track origin composition by jet flavour

Data loading first, then plotting cells that can be tweaked/re-run independently.

In [ ]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
# ── config ───────────────────────────────────────────────────────────
H5_FILE  = "mc-flavtag-ttbar-small.h5"
N_JETS   = 200_000
D0_CUT   = 3.5        # |d0| < D0_CUT to flag valid tracks
Z0_CUT   = 5.0        # |z0SinTheta| < Z0_CUT to flag valid tracks
PLOT_DIR = "./track_origin_plots/"

FLAVOUR_TO_LABEL = {5: 0, 4: 1, 0: 2}
CLASS_NAMES      = ["b-jet", "c-jet", "light-jet"]
FLAVOUR_COLOURS  = ["#1f77b4", "#ff7f0e", "#2ca02c"]

ORIGIN_NAMES = [
    "Pileup",      # 0
    "Fake",        # 1
    "Primary",     # 2
    "From b",      # 3
    "From b\u2192c",    # 4
    "From c",      # 5
    "From \u03c4",      # 6
    "Other sec.",  # 7
]
N_ORIGINS = 8

os.makedirs(PLOT_DIR, exist_ok=True)

## Load data

In [ ]:
print(f"Loading {N_JETS:,} jets from {H5_FILE}...")
with h5py.File(H5_FILE, "r") as f:
    all_flavour = f["jets"]["HadronConeExclTruthLabelID"][:N_JETS]
    keep_jet    = np.isin(all_flavour, list(FLAVOUR_TO_LABEL.keys()))
    jet_idx     = np.where(keep_jet)[0]

    flavour_label = np.array([FLAVOUR_TO_LABEL[v] for v in all_flavour[keep_jet]])

    valid       = f["tracks"]["valid"][jet_idx]                          # (N, K) bool
    d0          = f["tracks"]["d0"][jet_idx].astype(np.float32)
    z0sintheta  = f["tracks"]["z0SinTheta"][jet_idx].astype(np.float32)
    origin      = f["tracks"]["GN2v01_trackOrigin"][jet_idx].astype(np.int8)

N, K = valid.shape
print(f"Jets kept: {N:,}  |  tracks per jet (K): {K}")

In [ ]:
# Valid track mask: flagged valid + |d0| < D0_CUT + |z0SinTheta| < Z0_CUT
track_valid = valid & (np.abs(d0) < D0_CUT) & (np.abs(z0sintheta) < Z0_CUT)   # (N, K)

# Flatten everything to 1-D arrays
flat_valid  = track_valid.ravel()                     # (N*K,)
flat_origin = origin.ravel()                          # (N*K,)
flat_flav   = np.repeat(flavour_label, K)             # (N*K,)

## Compute origin fractions per flavour

In [ ]:
# Compute fraction of valid tracks from each origin, per jet flavour.
origin_counts = np.zeros((3, N_ORIGINS), dtype=np.int64)
for flav_idx in range(3):
    mask = flat_valid & (flat_flav == flav_idx)
    for orig_idx in range(N_ORIGINS):
        origin_counts[flav_idx, orig_idx] = (mask & (flat_origin == orig_idx)).sum()

totals    = origin_counts.sum(axis=1, keepdims=True)          # (3, 1)
fractions = origin_counts / np.where(totals > 0, totals, 1)   # (3, 8)

cmap_orig    = plt.get_cmap("tab10")
origin_cols  = [cmap_orig(i) for i in range(N_ORIGINS)]

fractions

## Plot 1: stacked bar per flavour

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

x  = np.arange(3)
bottoms = np.zeros(3)
for orig_idx in range(N_ORIGINS):
    frac = fractions[:, orig_idx]
    ax.bar(x, frac, bottom=bottoms, color=origin_cols[orig_idx],
           label=f"{orig_idx}: {ORIGIN_NAMES[orig_idx]}", edgecolor="white", linewidth=0.4)
    # annotate segments that are large enough to read
    for xi, (bot, f) in enumerate(zip(bottoms, frac)):
        if f > 0.03:
            ax.text(xi, bot + f / 2, f"{f:.1%}", ha="center", va="center",
                    fontsize=7, color="white", fontweight="bold")
    bottoms += frac

ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, fontsize=12)
ax.set_ylabel("Fraction of valid tracks", fontsize=12)
ax.set_ylim(0, 1.2)
ax.legend(ncol=4, fontsize=12, loc="upper center")

plt.tight_layout()
out = os.path.join(PLOT_DIR, "track_origin_fractions_stacked.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
print(f"Saved {out}")
plt.show()

## Plot 2: grouped bars — one group per origin, bars = flavours

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

w    = 0.22
offsets = np.array([-w, 0, w])
ox   = np.arange(N_ORIGINS)
for flav_idx, (flav_name, colour) in enumerate(zip(CLASS_NAMES, FLAVOUR_COLOURS)):
    ax.bar(ox + offsets[flav_idx], fractions[flav_idx], width=w,
           color=colour, label=flav_name, edgecolor="white", linewidth=0.4)

ax.set_xticks(ox)
ax.set_xticklabels(
    [f"{i}\n{ORIGIN_NAMES[i]}" for i in range(N_ORIGINS)],
    fontsize=12,
)
ax.set_ylabel("Fraction of valid tracks", fontsize=12)
ax.legend(fontsize=12)
ax.set_ylim(0, fractions.max() * 1.15)

plt.tight_layout()
out = os.path.join(PLOT_DIR, "track_origin_fractions_grouped.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
print(f"Saved {out}")
plt.show()